# NB_11 — Stage 11: Language Model Perplexity Analysis

**Purpose:** Measure whether transcription errors in Qwen and EasyOCR outputs produce
linguistically unnatural Arabic text, not just character-level mistakes.

**What perplexity measures:**  
An n-gram language model assigns probabilities to word sequences based on how
often those sequences appear in training text. Perplexity measures how surprised
the model is by a new piece of text. Low perplexity = natural, fluent text that
follows expected patterns. High perplexity = unnatural sequences, likely caused
by OCR errors producing garbled words that no language model would predict.

**Why this matters for our pipeline:**  
CER measures character-level agreement with the reference. Perplexity measures
linguistic naturalness independently of the reference. If a model produces text
with high CER AND high perplexity, its errors are linguistically disruptive.
If CER is high but perplexity is low, the model is producing plausible Arabic
that just happens to differ from the reference — a qualitatively different kind
of error.

**Design:**  
GPT transcriptions are the closest thing to ground truth available. We split them
200/80: the first 200 are used to train bigram and trigram language models; the
last 80 are the held-out evaluation set. All four sources (GPT, Qwen fine-tuned,
Qwen zero-shot, EasyOCR) are evaluated on the same 80-sample set so no model
has seen the evaluation text during LM training.

**No GPU required. Runs in under 2 minutes.**

**Prerequisites:**
- `data/eval/eval.jsonl`
- `logs/run-1/eval_results_checkpoint-1120.json` (with `per_sample`)
- `logs/run-1/eval_results_zero_shot.json` (with `per_sample`)
- `logs/stage7/easyocr_results.json`

## Step 11.1 — Mount Drive and set paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project'

EVAL_JSONL         = f'{PROJECT_ROOT}/data/eval/eval.jsonl'
QWEN_EVAL_FILE     = f'{PROJECT_ROOT}/logs/run-1/eval_results_checkpoint-1120.json'
ZEROSHOT_EVAL_FILE = f'{PROJECT_ROOT}/logs/run-1/eval_results_zero_shot.json'
EASYOCR_FILE       = f'{PROJECT_ROOT}/logs/stage7/easyocr_results.json'
RESULTS_DIR        = f'{PROJECT_ROOT}/logs/stage11'
os.makedirs(RESULTS_DIR, exist_ok=True)

# LM split sizes
LM_TRAIN_SIZE = 200   # GPT transcriptions used to build the n-gram LMs
LM_EVAL_SIZE  = 80    # held-out samples evaluated for all four sources

missing = []
for label, path in [
    ('Eval JSONL',             EVAL_JSONL),
    ('Qwen eval results',      QWEN_EVAL_FILE),
    ('Zero-shot eval results', ZEROSHOT_EVAL_FILE),
    ('EasyOCR results',        EASYOCR_FILE),
]:
    exists = os.path.exists(path)
    print(f'  {"✓" if exists else "✗"} {label}')
    if not exists:
        missing.append(label)

if missing:
    raise FileNotFoundError(f'Missing: {missing}. Run NB_06, NB_08, NB_09 first.')
print('\nAll input files found.')

Mounted at /content/drive
  ✓ Eval JSONL
  ✓ Qwen eval results
  ✓ Zero-shot eval results
  ✓ EasyOCR results

All input files found.


## Step 11.2 — Install dependencies

`nltk` is pre-installed on Colab. We use `nltk.lm` for n-gram language models
with Laplace (add-one) smoothing, which prevents zero-probability assignments
for unseen word sequences — a critical requirement for perplexity computation
on held-out text.

In [3]:
import subprocess, sys, importlib.metadata

# jiwer for CER correlation analysis
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'jiwer', '--quiet'],
    check=True
)

import nltk
nltk.download('punkt',    quiet=True)
nltk.download('punkt_tab', quiet=True)

print(f'nltk  version : {importlib.metadata.version("nltk")}')
print(f'jiwer version : {importlib.metadata.version("jiwer")}')
print('Dependencies ready.')

nltk  version : 3.9.1
jiwer version : 4.0.0
Dependencies ready.


## Step 11.3 — Load and tokenize all transcriptions

Arabic tokenization is done by whitespace splitting. This is sufficient for
perplexity comparison because all four sources (GPT, Qwen, zero-shot, EasyOCR)
are tokenized the same way, so comparisons are fair. A morphological tokenizer
would give more linguistically precise results but is not needed here given the
camel-tools compatibility issues documented in NB_07.

In [6]:
import json, re
def tokenize_arabic(text: str) -> list:
    """Whitespace tokenization with basic cleaning.
    Returns a list of non-empty tokens."""
    if not text or not isinstance(text, str):
        return []
    # Remove purely punctuation-only tokens but keep Arabic words
    tokens = text.split()
    tokens = [t.strip('.,،؛؟!()[]"\'-') for t in tokens]
    tokens = [t for t in tokens if t and re.search(r'[\u0600-\u06FF]', t)]
    return tokens

In [7]:
# ── GPT references from eval.jsonl ────────────────────────────────────────
gpt_texts = []
with open(EVAL_JSONL) as f:
    for line in f:
        sample = json.loads(line)
        for msg in sample['messages']:
            if msg['role'] == 'assistant':
                try:
                    parsed = json.loads(msg['content'])
                    text = parsed.get('transcription', '').strip()
                    if text:
                        gpt_texts.append(text)
                except Exception:
                    pass

assert len(gpt_texts) == 280, f'Expected 280 GPT refs, got {len(gpt_texts)}'
print(f'GPT transcriptions loaded: {len(gpt_texts)}')

GPT transcriptions loaded: 280


In [14]:
import json
with open(QWEN_EVAL_FILE) as f:
    qwen_data = json.load(f)
sample = qwen_data['per_sample'][0]
print('Qwen per_sample keys:', list(sample.keys()))
print('Sample:', {k: str(v)[:50] for k, v in sample.items()})

Qwen per_sample keys: ['idx', 'reference', 'hypothesis', 'is_valid_json', 'inference_time', 'raw_output']
Sample: {'idx': '0', 'reference': 'بلفات اليمين القديمة.', 'hypothesis': 'لبنات السمن القديمة.', 'is_valid_json': 'True', 'inference_time': '3.878214120864868', 'raw_output': 'لبنات السمن القديمة."}'}


In [17]:
from jiwer import cer as jiwer_cer

with open(QWEN_EVAL_FILE) as f:
    qwen_data = json.load(f)

if 'per_sample' not in qwen_data:
    raise KeyError(f'per_sample missing. Keys: {list(qwen_data.keys())}')

qwen_all = qwen_data['per_sample']

qwen_eval_texts = []
qwen_eval_refs  = []
qwen_eval_cers  = []

for s in qwen_all[LM_TRAIN_SIZE:]:
    hyp = s.get('hypothesis', '').strip()
    ref = s.get('reference',  '').strip()
    if not hyp or not ref:
        continue
    try:
        cer_val = float(jiwer_cer(ref, hyp))
    except Exception:
        cer_val = 1.0
    qwen_eval_texts.append(hyp)
    qwen_eval_refs.append(ref)
    qwen_eval_cers.append(cer_val)

print(f'Qwen fine-tuned eval samples: {len(qwen_eval_texts)}')
print(f'Mean CER (computed): {sum(qwen_eval_cers)/len(qwen_eval_cers):.4f}')

Qwen fine-tuned eval samples: 80
Mean CER (computed): 0.3177


In [19]:
with open(ZEROSHOT_EVAL_FILE) as f:
    zs_data = json.load(f)

if 'per_sample' not in zs_data:
    raise KeyError(f'per_sample missing. Keys: {list(zs_data.keys())}')

zs_all = zs_data['per_sample']

zs_eval_texts = []
zs_eval_cers  = []

for s in zs_all[LM_TRAIN_SIZE:]:
    # NB_09 used 'prediction' not 'hypothesis'
    hyp = s.get('prediction', s.get('hypothesis', '')).strip()
    ref = s.get('reference', '').strip()
    if not hyp or not ref:
        continue
    cer_val = s.get('cer', None)
    if cer_val is None:
        try:
            cer_val = float(jiwer_cer(ref, hyp))
        except Exception:
            cer_val = 1.0
    zs_eval_texts.append(hyp)
    zs_eval_cers.append(float(cer_val))

print(f'Zero-shot Qwen eval samples: {len(zs_eval_texts)}')
print(f'Mean CER (from saved or computed): {sum(zs_eval_cers)/len(zs_eval_cers):.4f}')

Zero-shot Qwen eval samples: 80
Mean CER (from saved or computed): 0.4643


In [11]:
# ── EasyOCR (10 stored samples — used as-is, indices don't align) ─────────
with open(EASYOCR_FILE) as f:
    easyocr_data = json.load(f)

easyocr_eval_texts = [
    s['predicted'] for s in easyocr_data.get('sample_outputs', [])
    if s.get('predicted', '').strip()
]
easyocr_eval_refs = [
    s['reference'] for s in easyocr_data.get('sample_outputs', [])
    if s.get('predicted', '').strip()
]
print(f'EasyOCR eval samples: {len(easyocr_eval_texts)} (first 10 only)')

EasyOCR eval samples: 10 (first 10 only)


In [20]:
# ── Apply 200/80 split to GPT texts ──────────────────────────────────────
gpt_lm_train = gpt_texts[:LM_TRAIN_SIZE]   # used to build LMs
gpt_lm_eval  = gpt_texts[LM_TRAIN_SIZE:]   # held-out evaluation

print(f'\nLM training set (GPT): {len(gpt_lm_train)} transcriptions')
print(f'LM eval set (all sources): {len(gpt_lm_eval)} transcriptions')

# Tokenize everything
gpt_train_tokens  = [tokenize_arabic(t) for t in gpt_lm_train]
gpt_eval_tokens   = [tokenize_arabic(t) for t in gpt_lm_eval]
qwen_eval_tokens  = [tokenize_arabic(t) for t in qwen_eval_texts]
zs_eval_tokens    = [tokenize_arabic(t) for t in zs_eval_texts]
easy_eval_tokens  = [tokenize_arabic(t) for t in easyocr_eval_texts]

# Filter out empty tokenized sequences
gpt_train_tokens = [t for t in gpt_train_tokens if len(t) >= 2]
print(f'\nNon-empty training sequences: {len(gpt_train_tokens)}')
print(f'Sample tokenized sentence: {gpt_train_tokens[0]}')


LM training set (GPT): 200 transcriptions
LM eval set (all sources): 80 transcriptions

Non-empty training sequences: 197
Sample tokenized sentence: ['بلفات', 'اليمين', 'القديمة']


## Step 11.4 — Build bigram and trigram language models

We use NLTK's `MLE` (Maximum Likelihood Estimation) with `Laplace` smoothing.
Laplace adds a count of 1 to every possible n-gram, which prevents zero
probability for unseen sequences — essential for computing finite perplexity
on held-out text that may contain words not seen in training.

Both models are trained on the same 200 GPT transcriptions.

In [21]:
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline


def build_ngram_lm(tokenized_sentences: list, n: int):
    """
    Build an n-gram language model with Laplace smoothing.
    Returns the fitted model.
    """
    train_data, vocab = padded_everygram_pipeline(n, tokenized_sentences)
    model = Laplace(n)
    model.fit(train_data, vocab)
    return model


print('Building bigram LM (n=2)...')
bigram_lm  = build_ngram_lm(gpt_train_tokens, n=2)
print(f'  Vocabulary size: {len(bigram_lm.vocab):,} tokens')

print('Building trigram LM (n=3)...')
trigram_lm = build_ngram_lm(gpt_train_tokens, n=3)
print(f'  Vocabulary size: {len(trigram_lm.vocab):,} tokens')

print('\nLMs built successfully.')

Building bigram LM (n=2)...
  Vocabulary size: 1,530 tokens
Building trigram LM (n=3)...
  Vocabulary size: 1,530 tokens

LMs built successfully.


## Step 11.5 — Define perplexity computation

NLTK's `model.perplexity()` requires the text to be padded and segmented into
n-grams using the same pipeline as training. We compute perplexity per sentence
and then average, rather than computing over the concatenated corpus, so that
sentence length does not bias the result — shorter sentences tend to have higher
perplexity because there is less context for the model to work with.

In [22]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import ngrams
from nltk.lm.preprocessing import pad_both_ends
import math


def sentence_perplexity(model, tokens: list, n: int) -> float:
    """
    Compute perplexity of a single tokenized sentence under an n-gram model.
    Returns float('inf') for empty or too-short sequences.
    """
    if len(tokens) < n:
        return float('inf')

    padded = list(pad_both_ends(tokens, n=n))
    test_ngrams = list(ngrams(padded, n))

    if not test_ngrams:
        return float('inf')

    try:
        return model.perplexity(test_ngrams)
    except Exception:
        return float('inf')


def corpus_perplexity(model, tokenized_sentences: list, n: int,
                      label: str = '') -> dict:
    """
    Compute mean and median perplexity over a list of tokenized sentences.
    Sentences with infinite perplexity (empty or too short) are excluded
    from the mean but counted and reported.
    """
    scores = []
    inf_count = 0

    for tokens in tokenized_sentences:
        ppl = sentence_perplexity(model, tokens, n)
        if math.isinf(ppl) or math.isnan(ppl):
            inf_count += 1
        else:
            scores.append(ppl)

    if not scores:
        return {'mean': float('inf'), 'median': float('inf'),
                'min': float('inf'), 'max': float('inf'),
                'n_valid': 0, 'n_inf': inf_count, 'scores': []}

    scores_sorted = sorted(scores)
    mid = len(scores_sorted) // 2
    median = (
        scores_sorted[mid]
        if len(scores_sorted) % 2 == 1
        else (scores_sorted[mid - 1] + scores_sorted[mid]) / 2
    )

    return {
        'mean':    round(sum(scores) / len(scores), 2),
        'median':  round(median, 2),
        'min':     round(min(scores), 2),
        'max':     round(max(scores), 2),
        'n_valid': len(scores),
        'n_inf':   inf_count,
        'scores':  scores,
    }


print('Perplexity functions defined.')

Perplexity functions defined.


## Step 11.6 — Compute perplexity for all sources

Each source is evaluated under both the bigram and trigram LMs.
Expected ordering (lowest to highest perplexity):
GPT < Qwen fine-tuned < Qwen zero-shot < EasyOCR

GPT should score lowest because the LM was trained on GPT text.
EasyOCR should score highest because OCR errors produce garbled words
that never appeared in training and have unpredictable n-gram contexts.

In [23]:
sources = [
    ('GPT (reference)',                     gpt_eval_tokens,  'reference'),
    ('Qwen2.5-VL-7B + LoRA (ckpt-1120)',   qwen_eval_tokens,  'fine-tuned'),
    ('Qwen2.5-VL-7B zero-shot',            zs_eval_tokens,    'zero-shot'),
    ('EasyOCR (Arabic, 10 samples)',        easy_eval_tokens,  'easyocr'),
]

bigram_results  = {}
trigram_results = {}

for label, tokens, key in sources:
    bg = corpus_perplexity(bigram_lm,  tokens, n=2, label=label)
    tg = corpus_perplexity(trigram_lm, tokens, n=3, label=label)
    bigram_results[key]  = {**bg,  'label': label}
    trigram_results[key] = {**tg,  'label': label}

    print(f'\n{label} (n={bg["n_valid"]} valid sequences)')
    print(f'  Bigram  — mean: {bg["mean"]:>10.2f}   median: {bg["median"]:>10.2f}')
    print(f'  Trigram — mean: {tg["mean"]:>10.2f}   median: {tg["median"]:>10.2f}')
    if bg['n_inf'] > 0:
        print(f'  ({bg["n_inf"]} sequences too short for reliable scoring — excluded)')


GPT (reference) (n=79 valid sequences)
  Bigram  — mean:    1504.61   median:    1547.12
  Trigram — mean:    1510.74   median:    1543.29
  (1 sequences too short for reliable scoring — excluded)

Qwen2.5-VL-7B + LoRA (ckpt-1120) (n=79 valid sequences)
  Bigram  — mean:    1509.26   median:    1547.40
  Trigram — mean:    1516.64   median:    1543.29
  (1 sequences too short for reliable scoring — excluded)

Qwen2.5-VL-7B zero-shot (n=79 valid sequences)
  Bigram  — mean:    1513.17   median:    1546.68
  Trigram — mean:    1518.97   median:    1543.29
  (1 sequences too short for reliable scoring — excluded)

EasyOCR (Arabic, 10 samples) (n=10 valid sequences)
  Bigram  — mean:    1504.33   median:    1543.40
  Trigram — mean:    1512.72   median:    1542.40


## Step 11.7 — Summary table

Print the consolidated comparison table for the paper.
We report mean perplexity because it is the standard metric,
and median alongside it because perplexity distributions are
right-skewed — a few very high-perplexity samples (severely
garbled OCR outputs) can inflate the mean considerably.

In [24]:
print('=' * 75)
print('PERPLEXITY SUMMARY TABLE')
print('LM trained on 200 GPT transcriptions | Evaluated on 80 held-out samples')
print('=' * 75)
print(f'{"Source":<40} {"Bigram PPL":>12} {"Trigram PPL":>12} {"n":>5}')
print('-' * 75)

order = ['reference', 'fine-tuned', 'zero-shot', 'easyocr']
for key in order:
    bg = bigram_results[key]
    tg = trigram_results[key]
    print(
        f'{bg["label"]:<40} '
        f'{bg["mean"]:>12.2f} '
        f'{tg["mean"]:>12.2f} '
        f'{bg["n_valid"]:>5}'
    )

print('=' * 75)
print('PPL = perplexity (lower = more natural Arabic text)')
print('n   = number of valid sequences (sequences ≥ n words)')

PERPLEXITY SUMMARY TABLE
LM trained on 200 GPT transcriptions | Evaluated on 80 held-out samples
Source                                     Bigram PPL  Trigram PPL     n
---------------------------------------------------------------------------
GPT (reference)                               1504.61      1510.74    79
Qwen2.5-VL-7B + LoRA (ckpt-1120)              1509.26      1516.64    79
Qwen2.5-VL-7B zero-shot                       1513.17      1518.97    79
EasyOCR (Arabic, 10 samples)                  1504.33      1512.72    10
PPL = perplexity (lower = more natural Arabic text)
n   = number of valid sequences (sequences ≥ n words)


## Step 11.8 — Correlation between perplexity and CER

Compute Pearson and Spearman correlations between per-sample perplexity
and per-sample CER for Qwen fine-tuned and zero-shot.

A strong positive correlation means: samples where the model makes more
transcription errors (high CER) also produce more linguistically unnatural
text (high perplexity). This would confirm that errors are not just
character-level mismatches but genuine linguistic disruptions.

A weak correlation would mean: CER errors are mostly orthographic
(different but valid Arabic spellings) rather than garbled text.

In [25]:
import math


def pearson_correlation(x: list, y: list) -> float:
    """Compute Pearson r between two equal-length lists."""
    n = len(x)
    if n < 2:
        return float('nan')
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    num    = sum((a - mean_x) * (b - mean_y) for a, b in zip(x, y))
    den_x  = math.sqrt(sum((a - mean_x) ** 2 for a in x))
    den_y  = math.sqrt(sum((b - mean_y) ** 2 for b in y))
    if den_x == 0 or den_y == 0:
        return float('nan')
    return round(num / (den_x * den_y), 4)


def spearman_correlation(x: list, y: list) -> float:
    """Compute Spearman rank correlation between two equal-length lists."""
    n = len(x)
    if n < 2:
        return float('nan')

    def rank(lst):
        sorted_lst = sorted(enumerate(lst), key=lambda t: t[1])
        ranks = [0] * n
        for rank_val, (idx, _) in enumerate(sorted_lst, 1):
            ranks[idx] = rank_val
        return ranks

    return pearson_correlation(rank(x), rank(y))


def compute_correlation(model, token_list, cer_list, n, label):
    """Pair per-sample perplexity with CER, filter infinities, correlate."""
    ppls = [sentence_perplexity(model, t, n) for t in token_list]

    # Keep only pairs where both are finite
    pairs = [
        (ppl, cer_val)
        for ppl, cer_val in zip(ppls, cer_list)
        if not math.isinf(ppl) and not math.isnan(ppl)
    ]

    if len(pairs) < 5:
        print(f'{label}: not enough valid pairs for correlation ({len(pairs)})')
        return None

    ppls_clean, cers_clean = zip(*pairs)
    pearson  = pearson_correlation(list(ppls_clean),  list(cers_clean))
    spearman = spearman_correlation(list(ppls_clean), list(cers_clean))

    print(f'{label} (n={len(pairs)} pairs, n={n}-gram):')
    print(f'  Pearson r  = {pearson:>7.4f}')
    print(f'  Spearman ρ = {spearman:>7.4f}')

    return {
        'n_pairs':   len(pairs),
        'pearson_r': pearson,
        'spearman':  spearman,
        'ppls':      list(ppls_clean),
        'cers':      list(cers_clean),
    }


print('=== Perplexity–CER Correlation ===')
print()

qwen_bg_corr  = compute_correlation(bigram_lm,  qwen_eval_tokens, qwen_eval_cers, 2, 'Qwen fine-tuned  (bigram)')
qwen_tg_corr  = compute_correlation(trigram_lm, qwen_eval_tokens, qwen_eval_cers, 3, 'Qwen fine-tuned  (trigram)')
print()
zs_bg_corr    = compute_correlation(bigram_lm,  zs_eval_tokens, zs_eval_cers, 2, 'Qwen zero-shot   (bigram)')
zs_tg_corr    = compute_correlation(trigram_lm, zs_eval_tokens, zs_eval_cers, 3, 'Qwen zero-shot   (trigram)')

=== Perplexity–CER Correlation ===

Qwen fine-tuned  (bigram) (n=79 pairs, n=2-gram):
  Pearson r  = -0.1944
  Spearman ρ = -0.1077
Qwen fine-tuned  (trigram) (n=78 pairs, n=3-gram):
  Pearson r  = -0.2689
  Spearman ρ = -0.0726

Qwen zero-shot   (bigram) (n=79 pairs, n=2-gram):
  Pearson r  =  0.1860
  Spearman ρ =  0.2014
Qwen zero-shot   (trigram) (n=78 pairs, n=3-gram):
  Pearson r  =  0.0919
  Spearman ρ =  0.1259


## Step 11.9 — Worst perplexity examples

Identify the 5 Qwen fine-tuned outputs with the highest bigram perplexity.
These are the samples where the model produced the most linguistically
unnatural Arabic — the qualitative complement to the quantitative CER analysis.

In [26]:
# Compute per-sample bigram perplexity for Qwen fine-tuned eval set
qwen_sample_ppls = [
    (sentence_perplexity(bigram_lm, tokens, 2), ref, hyp, cer_val)
    for tokens, ref, hyp, cer_val in zip(
        qwen_eval_tokens,
        qwen_eval_refs,
        qwen_eval_texts,
        qwen_eval_cers
    )
    if not math.isinf(sentence_perplexity(bigram_lm, tokens, 2))
]

qwen_sample_ppls.sort(key=lambda x: -x[0])

print('=== Top 5 Highest Perplexity (Qwen fine-tuned, bigram LM) ===')
for rank, (ppl, ref, hyp, cer_val) in enumerate(qwen_sample_ppls[:5], 1):
    print(f'\n[{rank}] Perplexity: {ppl:.2f}  |  CER: {cer_val:.4f}')
    print(f'  REF: {ref}')
    print(f'  HYP: {hyp}')

print('\n=== Top 5 Lowest Perplexity (most natural outputs) ===')
for rank, (ppl, ref, hyp, cer_val) in enumerate(qwen_sample_ppls[-5:][::-1], 1):
    print(f'\n[{rank}] Perplexity: {ppl:.2f}  |  CER: {cer_val:.4f}')
    print(f'  REF: {ref}')
    print(f'  HYP: {hyp}')

=== Top 5 Highest Perplexity (Qwen fine-tuned, bigram LM) ===

[1] Perplexity: 1594.52  |  CER: 0.2941
  REF: التأثر في المجتمع
  HYP: الانتشار في المجتمع.

[2] Perplexity: 1593.03  |  CER: 0.5000
  REF: ان يقبسه
  HYP: الن يقصه.

[3] Perplexity: 1577.04  |  CER: 0.2667
  REF: وعشره تحت الصفر
  HYP: وعشرية كت الصفر

[4] Perplexity: 1573.55  |  CER: 0.2000
  REF: هاملك من الاحترام للمرأة.
  HYP: هما كلًا من الاحترام للمرأة.

[5] Perplexity: 1568.13  |  CER: 0.0952
  REF: تاءً، وان باطن الأرض.
  HYP: تاماً، وان باطن الأرض.

=== Top 5 Lowest Perplexity (most natural outputs) ===

[1] Perplexity: 1247.43  |  CER: 0.9535
  REF: وقد زاد اتساعها المناطق الجنوبية الممتدة من
  HYP: وقد زارنا أنتا كما الناطقة الجوف ببيه الفرقة و للنقطة المنسوبة للمبدع من

[2] Perplexity: 1274.41  |  CER: 0.4259
  REF: في اصدار، الندوة، ضف عثر على أنس ناحيه - جلسله، أو ابن
  HYP: في أصدر المذورة كشفت على أننا ناحية جليلة، أو وانف

[3] Perplexity: 1336.57  |  CER: 0.2840
  REF: في هدوء مع جبهة العين اليوم قالت ابي

## Step 11.10 — Save all results

In [27]:
import json, os

def safe_corr(corr_dict):
    if corr_dict is None:
        return None
    return {
        'n_pairs':   corr_dict['n_pairs'],
        'pearson_r': corr_dict['pearson_r'],
        'spearman':  corr_dict['spearman'],
    }

output = {
    'lm_config': {
        'smoothing':       'Laplace (add-1)',
        'tokenization':    'whitespace',
        'train_size':      LM_TRAIN_SIZE,
        'eval_size':       LM_EVAL_SIZE,
        'train_source':    'GPT transcriptions (first 200 of eval.jsonl)',
    },
    'perplexity_table': {
        key: {
            'label':          bigram_results[key]['label'],
            'n_valid':        bigram_results[key]['n_valid'],
            'bigram_mean':    bigram_results[key]['mean'],
            'bigram_median':  bigram_results[key]['median'],
            'trigram_mean':   trigram_results[key]['mean'],
            'trigram_median': trigram_results[key]['median'],
        }
        for key in ['reference', 'fine-tuned', 'zero-shot', 'easyocr']
    },
    'cer_perplexity_correlation': {
        'qwen_finetuned_bigram':  safe_corr(qwen_bg_corr),
        'qwen_finetuned_trigram': safe_corr(qwen_tg_corr),
        'qwen_zeroshot_bigram':   safe_corr(zs_bg_corr),
        'qwen_zeroshot_trigram':  safe_corr(zs_tg_corr),
    },
    'top5_highest_perplexity_qwen': [
        {'perplexity': round(ppl, 2), 'cer': cer_val, 'reference': ref, 'hypothesis': hyp}
        for ppl, ref, hyp, cer_val in qwen_sample_ppls[:5]
    ],
    'top5_lowest_perplexity_qwen': [
        {'perplexity': round(ppl, 2), 'cer': cer_val, 'reference': ref, 'hypothesis': hyp}
        for ppl, ref, hyp, cer_val in qwen_sample_ppls[-5:][::-1]
    ],
    'note': (
        'N-gram LMs built with NLTK Laplace smoothing on 200 GPT transcriptions. '
        'Perplexity computed on held-out 80 samples. '
        'EasyOCR limited to 10 stored samples. '
        'Infinite-perplexity sentences (< n words) excluded from means.'
    ),
}

out_path = f'{RESULTS_DIR}/perplexity_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Results saved to {out_path}')
print('NB_11 complete.')

Results saved to /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/stage11/perplexity_results.json
NB_11 complete.
